In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RESULTS_DIR = Path("../results")

HANDCRAFTED = RESULTS_DIR / "handcrafted_loso_full.csv"
CNN = RESULTS_DIR / "cnn_full_modality_results.csv"
CNN_LSTM = RESULTS_DIR / "cnn_lstm_full_modality_results.csv"

In [3]:
handcrafted = pd.read_csv(HANDCRAFTED)
cnn = pd.read_csv(CNN)
cnn_lstm = pd.read_csv(CNN_LSTM)

print(handcrafted.head())
print(cnn.head())
print(cnn_lstm.head())

   subject    classifier  accuracy  macro_f1     auroc  n_test_windows  \
0        2  RandomForest  0.857143  0.805771  0.938901             140   
1        3  RandomForest  0.845070  0.778125  0.811190             142   
2        4  RandomForest  0.993007  0.991513  0.999044             143   
3        5  RandomForest  0.753425  0.674188  0.872940             146   
4        6  RandomForest  0.800000  0.718560  0.875171             145   

   n_train_windows  stress_ratio_test  
0             2047           0.292857  
1             2045           0.295775  
2             2044           0.286713  
3             2041           0.287671  
4             2042           0.296552  
   subject  val_subject    val_f1  accuracy  f1_macro     auroc  \
0        2           17  0.992298  0.771429  0.760428  0.895541   
1        3           17  1.000000  0.838028  0.765424  0.962143   
2        4           17  1.000000  0.811189  0.696104  0.999283   
3        5           17  0.992298  0.931507  0.

In [4]:
handcrafted_summary = (
    handcrafted
    .groupby("classifier")
    .agg(
        Accuracy_Mean=("accuracy", "mean"),
        Accuracy_SD=("accuracy", "std"),
        MacroF1_Mean=("macro_f1", "mean"),
        MacroF1_SD=("macro_f1", "std"),
        AUROC_Mean=("auroc", "mean"),
        AUROC_SD=("auroc", "std"),
    )
    .reset_index()
)

handcrafted_summary

,classifier,Accuracy_Mean,Accuracy_SD,MacroF1_Mean,MacroF1_SD,AUROC_Mean,AUROC_SD
0,RandomForest,0.880541,0.076317,0.854590,0.098143,0.957392,0.058521
1,SVM,0.895231,0.063064,0.874648,0.077102,0.972048,0.027214
2,XGBoost,0.881973,0.073035,0.846854,0.108827,0.972069,0.028948


In [7]:
print(handcrafted.columns)
print(cnn.columns)
print(cnn_lstm.columns)

Index(['subject', 'classifier', 'accuracy', 'macro_f1', 'auroc',
       'n_test_windows', 'n_train_windows', 'stress_ratio_test'],
      dtype='object')
Index(['subject', 'val_subject', 'val_f1', 'accuracy', 'f1_macro', 'auroc',
       'train_time_sec'],
      dtype='object')
Index(['subject', 'val_subject', 'val_f1', 'accuracy', 'f1_macro', 'auroc',
       'train_time_sec'],
      dtype='object')


In [10]:
cnn_summary = pd.DataFrame({
    "classifier": ["CNN"],
    "Accuracy_Mean": [cnn["accuracy"].mean()],
    "Accuracy_SD": [cnn["accuracy"].std()],
    "MacroF1_Mean": [cnn["f1_macro"].mean()],
    "MacroF1_SD": [cnn["f1_macro"].std()],
    "AUROC_Mean": [cnn["auroc"].mean()],
    "AUROC_SD": [cnn["auroc"].std()],
})

cnn_summary

,classifier,Accuracy_Mean,Accuracy_SD,MacroF1_Mean,MacroF1_SD,AUROC_Mean,AUROC_SD
0,CNN,0.893087,0.085514,0.849155,0.148726,0.970529,0.045845


In [11]:
cnn_lstm_summary = pd.DataFrame({
    "classifier": ["CNN-LSTM"],
    "Accuracy_Mean": [cnn_lstm["accuracy"].mean()],
    "Accuracy_SD": [cnn_lstm["accuracy"].std()],
    "MacroF1_Mean": [cnn_lstm["f1_macro"].mean()],
    "MacroF1_SD": [cnn_lstm["f1_macro"].std()],
    "AUROC_Mean": [cnn_lstm["auroc"].mean()],
    "AUROC_SD": [cnn_lstm["auroc"].std()],
})

cnn_lstm_summary

,classifier,Accuracy_Mean,Accuracy_SD,MacroF1_Mean,MacroF1_SD,AUROC_Mean,AUROC_SD
0,CNN-LSTM,0.896992,0.094076,0.878606,0.105155,0.948111,0.075903


In [12]:
final_table = pd.concat(
    [
        handcrafted_summary,
        cnn_summary,
        cnn_lstm_summary,
    ],
    ignore_index=True,
)

final_table

,classifier,Accuracy_Mean,Accuracy_SD,MacroF1_Mean,MacroF1_SD,AUROC_Mean,AUROC_SD
0,RandomForest,0.880541,0.076317,0.854590,0.098143,0.957392,0.058521
1,SVM,0.895231,0.063064,0.874648,0.077102,0.972048,0.027214
2,XGBoost,0.881973,0.073035,0.846854,0.108827,0.972069,0.028948
3,CNN,0.893087,0.085514,0.849155,0.148726,0.970529,0.045845
4,CNN-LSTM,0.896992,0.094076,0.878606,0.105155,0.948111,0.075903


In [13]:
final_table = final_table.sort_values(
    by="MacroF1_Mean",
    ascending=False
).reset_index(drop=True)

final_table

,classifier,Accuracy_Mean,Accuracy_SD,MacroF1_Mean,MacroF1_SD,AUROC_Mean,AUROC_SD
0,CNN-LSTM,0.896992,0.094076,0.878606,0.105155,0.948111,0.075903
1,SVM,0.895231,0.063064,0.874648,0.077102,0.972048,0.027214
2,RandomForest,0.880541,0.076317,0.854590,0.098143,0.957392,0.058521
3,CNN,0.893087,0.085514,0.849155,0.148726,0.970529,0.045845
4,XGBoost,0.881973,0.073035,0.846854,0.108827,0.972069,0.028948


In [14]:
final_table.insert(
    0,
    "Rank",
    range(1, len(final_table) + 1)
)

final_table

,Rank,classifier,Accuracy_Mean,Accuracy_SD,MacroF1_Mean,MacroF1_SD,AUROC_Mean,AUROC_SD
0,1,CNN-LSTM,0.896992,0.094076,0.878606,0.105155,0.948111,0.075903
1,2,SVM,0.895231,0.063064,0.874648,0.077102,0.972048,0.027214
2,3,RandomForest,0.880541,0.076317,0.854590,0.098143,0.957392,0.058521
3,4,CNN,0.893087,0.085514,0.849155,0.148726,0.970529,0.045845
4,5,XGBoost,0.881973,0.073035,0.846854,0.108827,0.972069,0.028948


In [15]:
paper_table = final_table.copy()

paper_table["Accuracy"] = (
    paper_table["Accuracy_Mean"].map("{:.4f}".format)
    + " ± "
    + paper_table["Accuracy_SD"].map("{:.4f}".format)
)

paper_table["Macro-F1"] = (
    paper_table["MacroF1_Mean"].map("{:.4f}".format)
    + " ± "
    + paper_table["MacroF1_SD"].map("{:.4f}".format)
)

paper_table["AUROC"] = (
    paper_table["AUROC_Mean"].map("{:.4f}".format)
    + " ± "
    + paper_table["AUROC_SD"].map("{:.4f}".format)
)

paper_table = paper_table[
    [
        "Rank",
        "classifier",
        "Accuracy",
        "Macro-F1",
        "AUROC",
    ]
]

paper_table

,Rank,classifier,Accuracy,Macro-F1,AUROC
0,1,CNN-LSTM,0.8970 ± 0.0941,0.8786 ± 0.1052,0.9481 ± 0.0759
1,2,SVM,0.8952 ± 0.0631,0.8746 ± 0.0771,0.9720 ± 0.0272
2,3,RandomForest,0.8805 ± 0.0763,0.8546 ± 0.0981,0.9574 ± 0.0585
3,4,CNN,0.8931 ± 0.0855,0.8492 ± 0.1487,0.9705 ± 0.0458
4,5,XGBoost,0.8820 ± 0.0730,0.8469 ± 0.1088,0.9721 ± 0.0289


In [16]:
OUTPUT = RESULTS_DIR / "full_modality_summary.csv"

paper_table.to_csv(
    OUTPUT,
    index=False,
)

print(f"Saved to {OUTPUT}")

Saved to ..\results\full_modality_summary.csv


In [17]:
latex = paper_table.to_latex(
    index=False,
    escape=False
)

print(latex)

\begin{tabular}{rllll}
\toprule
Rank & classifier & Accuracy & Macro-F1 & AUROC \\
\midrule
1 & CNN-LSTM & 0.8970 ± 0.0941 & 0.8786 ± 0.1052 & 0.9481 ± 0.0759 \\
2 & SVM & 0.8952 ± 0.0631 & 0.8746 ± 0.0771 & 0.9720 ± 0.0272 \\
3 & RandomForest & 0.8805 ± 0.0763 & 0.8546 ± 0.0981 & 0.9574 ± 0.0585 \\
4 & CNN & 0.8931 ± 0.0855 & 0.8492 ± 0.1487 & 0.9705 ± 0.0458 \\
5 & XGBoost & 0.8820 ± 0.0730 & 0.8469 ± 0.1088 & 0.9721 ± 0.0289 \\
\bottomrule
\end{tabular}

